# Creating Choropleth Map of Median AQI by US Counties

Needed data prep:

- AQI data ("annual_aqi_by_county_2026.csv") has county and state names, but doesn't have the FIPS codes for the corresponding state and county
- We can get corresponding FIPS codes from "aqs_sites.csv" file
- Then merge these 2 data sets to get a data set containing the median AQI and the FIPS codes
- FIPS code and a version of FIPS code without leading zero are needed to use charting libraries like Altair, Plotly, and Folium 

In [ ]:
import polars as pl

In [ ]:
df_aqi_by_counties = pl.read_csv(
    "../data/annual_aqi_by_county_2026.csv",
    columns=["State","County","Median AQI"],
    schema={
        "State": pl.String,
        "County": pl.String,
        "Median AQI": pl.Int64
    },
    extra_columns="ignore"
)

In [ ]:
df_aqi_by_counties.shape

In [ ]:
df_aqi_by_counties.head()

In [ ]:
df_aqs_sites = pl.read_csv(
    "../data/aqs_sites.csv",
    columns=["State Code","County Code","State Name","County Name"],
    schema={
        "State Code": pl.String,
        "County Code": pl.String,
        "State Name": pl.String,
        "County Name": pl.String
    },
    extra_columns="ignore"
)

In [ ]:
df_aqs_sites.head()

In [ ]:
df_fips_to_county_state_names = (
    df_aqs_sites.unique()
    .sort(pl.col("State Code"),pl.col("County Code"))
)

In [ ]:
df_fips_to_county_state_names.head()

#### Join the 2 data sets so that the FIPS code and median AQI are all in one data set (called df_joined)

In [ ]:
df_joined = df_aqi_by_counties.join(
    df_fips_to_county_state_names,
    how="left",
    left_on=["State","County"],
    right_on=["State Name","County Name"],
).sort(pl.col("State Code","County Code"))

In [ ]:
df_joined.head()

In [ ]:
assert df_joined.shape[0] == df_aqi_by_counties.shape[0]

#### Now create 5-digit FIPS code and id (5-digit FIPS without leading zero)

In [ ]:
df_joined = (
    df_joined
    .with_columns(
        fips=pl.col("State Code") + pl.col("County Code"),  # Create FIPS code so that we can use Plotly
        id=(pl.col("State Code") + pl.col("County Code")).cast(pl.Int64)   # Create id (FIPS without leading zero) so that we can use Altair
    )
)

In [ ]:
df_joined.head()

## Creating Choropleth Map Using Plotly

In [ ]:
import json
import urllib.request
import plotly.express as px

# County boundaries keyed by 5-digit FIPS
url = "https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json"
with urllib.request.urlopen(url) as f:
    counties = json.load(f)

fig = px.choropleth(
    df_joined,
    geojson=counties,
    # locations="id",
    locations="fips",
    color="Median AQI",
    color_continuous_scale="Viridis",
    # range_color=(0, 15),                # clip outliers so colors stay readable
    scope="usa",                          # Albers USA projection, AK/HI insets
    hover_name="State",
    hover_data={"County": True, "Median AQI": ":d"},
    # labels={"Unemploy-ment Rate (%)": "% Unemployed"},
    title="2026 Median AQI by US Counties",
)

fig.update_traces(marker_line_width=0.1)   # thin county borders
fig.update_layout(
    margin=dict(l=0, r=0, t=40, b=0),
    width=1200,
    height=600
)
fig.show()

From above, we see that counties with no data do not get rendered in the map.  Below is modified version that adds a background (all counties rendered) with the foreground added containing counties with data rendered.

In [ ]:
import json
import urllib.request
import plotly.express as px
import plotly.graph_objects as go

url = "https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json"
with urllib.request.urlopen(url) as f:
    counties = json.load(f)

# Every county FIPS in the geojson
all_fips = [feat["id"] for feat in counties["features"]]

fig = px.choropleth(
    df_joined,
    geojson=counties,
    locations="fips",
    color="Median AQI",
    color_continuous_scale="Viridis",
    scope="usa",
    hover_name="State",
    hover_data={"County": True, "Median AQI": ":d"},
    title="2026 Median AQI by US Counties",
)

# Background layer: all counties, flat gray, no hover
background = go.Choropleth(
    geojson=counties,
    locations=all_fips,
    z=[0] * len(all_fips),
    colorscale=[[0, "#e6e6e6"], [1, "#e6e6e6"]],
    showscale=False,
    marker_line_width=0.1,
    marker_line_color="black",
    hoverinfo="skip",
)

# Insert background first so the data layer draws on top
fig.add_trace(background)
# Trace order matters. Plotly draws traces in order, so the reordering line puts the gray layer underneath.
# Without it, the gray counties would cover your data.
fig.data = (fig.data[-1],) + fig.data[:-1]

fig.update_traces(marker_line_width=0.1, selector=dict(type="choropleth"))
fig.update_layout(
    margin=dict(l=0, r=0, t=40, b=0),
    width=1200,
    height=600,
)
fig.show()

## Creating Choropleth Map Using Altair

In [ ]:
import altair as alt
from altair.datasets import data

counties = alt.topo_feature(data.us_10m.url, "counties")

chart = (
    alt.Chart(counties)
    .mark_geoshape()
    .encode(
        color=alt.Color(
            "Median AQI:Q",
            title="Median AQI",
            scale=alt.Scale(
                scheme="viridis",
                # reverse=True,
                # domain=[1,15],
                # clamp=True
            ),
        ),
        tooltip=[
            alt.Tooltip("County:N", title="County"),
            alt.Tooltip("State:N", title="State"),
            alt.Tooltip("Median AQI:Q", title="Median AQI", format=".0f"),
        ],
    )
    .transform_lookup(
        lookup="id",
        from_=alt.LookupData(
            df_joined,
            "id",
            ["Median AQI", "County", "State"],
        ),
    )
    .project(type="albersUsa")
    .properties(width=800, height=500, title="2026 Median AQI of US Counties")
)

chart

From above, we see counties with no data are not rendered at all because of the transform_lookup() method is essentially doing an INNER JOIN for rendering: when the lookup finds no match, the fields come back null, and Vega-Lite drops marks whose encoded value is null or invalid. The fix is to layer a background map of all counties underneath, so every county outline is always drawn, then draw the data layer on top.

In [ ]:
import altair as alt
from altair.datasets import data

counties = alt.topo_feature(data.us_10m.url, "counties")

# Layer 1: every county, light gray fill + borders (no data needed)
background = (
    alt.Chart(counties)
    .mark_geoshape(fill="#e6e6e6", stroke="white", strokeWidth=0.3)
    .project(type="albersUsa")
)

# Layer 2: counties with data, colored by Median AQI
foreground = (
    alt.Chart(counties)
    .mark_geoshape(stroke="white", strokeWidth=0.3)
    .encode(
        color=alt.Color(
            "Median AQI:Q",
            title="Median AQI",
            scale=alt.Scale(scheme="viridis"),
        ),
        tooltip=[
            alt.Tooltip("County:N", title="County"),
            alt.Tooltip("State:N", title="State"),
            alt.Tooltip("Median AQI:Q", title="Median AQI", format=".0f"),
        ],
    )
    .transform_lookup(
        lookup="id",
        from_=alt.LookupData(df_joined, "id", ["Median AQI", "County", "State"]),
    )
    .project(type="albersUsa")
)

chart = (
    (background + foreground)
    .properties(width=800, height=500, title="2026 Median AQI of US Counties")
)

chart

## Creating Choropleth Map Using Folium

Unfortunately, when using polars, can't use the convenience function: folium.Choropleth(), which accepts pandas dataframe, not polars dataframe.  Workaround is to use GeoJson() method and a style method if hover tooltip is desired.  Therefore, Folium API is more verbose compared to Altair or Plotly for county-level choropleth map creation.

In [ ]:
import folium
import polars as pl
import requests
from branca.colormap import linear

geo_url = "https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json"
counties = requests.get(geo_url).json()

# FIPS -> value lookup, create a new attribue or key called "median_aqi" in the features dict and assign median AQI value to each id (or FIPS code) 
# State code -> state name lookup, create a new attribute called "STATE_NAME" in the features dict and assign the state name to each STATE FIPS code
median_aqi = dict(df_joined.select("fips", "Median AQI").iter_rows())
state_codes = dict(df_joined.select("State Code", "State").iter_rows())
for feature in counties["features"]:
    feature["properties"]["median_aqi"] = median_aqi.get(feature["id"])  # None if missing
    feature["properties"]["STATE_NAME"] = state_codes.get(feature["properties"]["STATE"])

# Color scale from the data range
colormap = linear.YlOrRd_09.scale(df_joined["Median AQI"].min(), df_joined["Median AQI"].max())
colormap.caption = "Median API"

def style_fn(feature):
    val = feature["properties"]["median_aqi"]
    return {
        "fillColor": colormap(val) if val is not None else "lightgray",
        "fillOpacity": 0.8,
        "color": "black",
        "weight": 0.2,
    }

def highlight_fn(feature):
    return {"weight": 2, "color": "black", "fillOpacity": 0.95}

m = folium.Map(location=[38, -96], zoom_start=4, tiles=None)

folium.GeoJson(
    counties,
    style_function=style_fn,
    highlight_function=highlight_fn,
    tooltip=folium.GeoJsonTooltip(
        fields=["STATE_NAME", "NAME", "median_aqi"],
        aliases=["State:", "County:", "Median AQI:"],
        localize=True,
    ),
).add_to(m)

colormap.add_to(m)  # legend
m